# AI Stock Agent — Demo Notebook

This notebook demonstrates the AI Stock Agent deployed on AWS Bedrock AgentCore.
It authenticates via Cognito JWT, invokes the agent through the AgentCore Gateway,
and streams responses via SSE for each of the 5 required queries.

**Prerequisites:**
- Infrastructure deployed via `terraform apply`
- Cognito test user created (see README)
- `pip install boto3 httpx`

## Cell 1: Configuration

Set the deployment parameters from Terraform outputs.

In [ ]:
import os

# ──────────────────────────────────────────────
# Fill these from `terraform output` or set as env vars
# ──────────────────────────────────────────────
GATEWAY_URL = os.environ.get("GATEWAY_URL", "https://<gateway-url>")
COGNITO_USER_POOL_ID = os.environ.get("COGNITO_USER_POOL_ID", "us-east-1_XXXXXXXXX")
COGNITO_CLIENT_ID = os.environ.get("COGNITO_CLIENT_ID", "xxxxxxxxxxxxxxxxxxxxxxxxxx")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

# Cognito test user credentials
COGNITO_USERNAME = os.environ.get("COGNITO_USERNAME", "testuser@example.com")
COGNITO_PASSWORD = os.environ.get("COGNITO_PASSWORD", "")

INVOCATIONS_URL = f"{GATEWAY_URL.rstrip('/')}/invocations"

print(f"Gateway URL:  {GATEWAY_URL}")
print(f"User Pool ID: {COGNITO_USER_POOL_ID}")
print(f"Client ID:    {COGNITO_CLIENT_ID}")
print(f"Region:       {AWS_REGION}")

## Cell 2: Helper Functions

- `authenticate()` — Obtain a JWT from Cognito via `InitiateAuth`
- `invoke_agent()` — POST to `/invocations` with SSE parsing and JWT header

In [ ]:
from __future__ import annotations

import json
import uuid

import boto3
import httpx


def authenticate(
    user_pool_id: str = COGNITO_USER_POOL_ID,
    client_id: str = COGNITO_CLIENT_ID,
    username: str = COGNITO_USERNAME,
    password: str = COGNITO_PASSWORD,
    region: str = AWS_REGION,
) -> dict:
    """Authenticate with Cognito and return the full auth result including tokens."""
    client = boto3.client("cognito-idp", region_name=region)
    response = client.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={
            "USERNAME": username,
            "PASSWORD": password,
        },
    )
    return response["AuthenticationResult"]


def invoke_agent(
    prompt: str,
    id_token: str,
    thread_id: str | None = None,
    stream: bool = True,
    url: str = INVOCATIONS_URL,
    timeout: float = 120.0,
) -> str:
    """Invoke the agent via the Gateway and print streaming tokens.

    Returns the full concatenated response text.
    """
    thread_id = thread_id or str(uuid.uuid4())
    payload = {"prompt": prompt, "thread_id": thread_id, "stream": stream}
    headers = {
        "Authorization": f"Bearer {id_token}",
        "Content-Type": "application/json",
    }

    full_response = []

    with httpx.Client(timeout=timeout) as client:
        with client.stream("POST", url, json=payload, headers=headers) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines():
                if not line.startswith("data: "):
                    continue
                data = json.loads(line[6:])
                if data.get("type") == "token":
                    token = data["content"]
                    print(token, end="", flush=True)
                    full_response.append(token)
                elif data.get("type") == "end":
                    break

    print()  # newline after streaming
    return "".join(full_response)


print("Helpers loaded ✓")

## Cell 3: Authenticate with Cognito

Obtain a JWT via `USER_PASSWORD_AUTH` flow.

In [ ]:
auth_result = authenticate()
id_token = auth_result["IdToken"]

print(f"Access Token (first 40 chars): {auth_result['AccessToken'][:40]}...")
print(f"ID Token     (first 40 chars): {id_token[:40]}...")
print(f"Token Type: {auth_result['TokenType']}")
print(f"Expires In: {auth_result['ExpiresIn']}s")
print("\nAuthentication successful ✓")

## Cell 4: Query 1 — Real-Time Stock Price

> *"What is the stock price for Amazon right now?"*

Expected: Agent calls `retrieve_realtime_stock_price` with ticker AMZN.

In [ ]:
response_1 = invoke_agent(
    "What is the stock price for Amazon right now?",
    id_token=id_token,
)

## Cell 5: Query 2 — Historical Stock Prices

> *"What were the stock prices for Amazon in Q4 last year?"*

Expected: Agent calls `retrieve_historical_stock_price` with Q4 2025 date range.

In [ ]:
response_2 = invoke_agent(
    "What were the stock prices for Amazon in Q4 last year?",
    id_token=id_token,
)

## Cell 6: Query 3 — Cross-Reference Stock + Reports

> *"Compare Amazon's recent stock performance to what analysts predicted in their reports"*

Expected: Agent calls both yfinance (historical) and RAG (earnings reports).

In [ ]:
response_3 = invoke_agent(
    "Compare Amazon's recent stock performance to what analysts predicted in their reports",
    id_token=id_token,
)

## Cell 7: Query 4 — Multi-Source Research

> *"I'm researching AMZN -- give me the current price and any relevant information about their AI business"*

Expected: Agent calls realtime tool + RAG tool, combines price with AI business insights.

In [ ]:
response_4 = invoke_agent(
    "I'm researching AMZN -- give me the current price and any relevant information about their AI business",
    id_token=id_token,
)

## Cell 8: Query 5 — Document-Only Query

> *"What is the total amount of office space Amazon owned in North America in 2024?"*

Expected: Agent calls RAG tool only, retrieves data from Amazon 2024 Annual Report.

In [ ]:
response_5 = invoke_agent(
    "What is the total amount of office space Amazon owned in North America in 2024?",
    id_token=id_token,
)

## Cell 9: Multi-Turn Demo

Two messages in the same `thread_id` to demonstrate conversation memory
via `AgentCoreMemorySaver`.

In [ ]:
import uuid

thread_id = str(uuid.uuid4())
print(f"Thread ID: {thread_id}")
print("="*60)

print("\n--- Turn 1 ---")
print("User: What is the current stock price of Amazon?\n")
turn_1 = invoke_agent(
    "What is the current stock price of Amazon?",
    id_token=id_token,
    thread_id=thread_id,
)

print("\n--- Turn 2 (follow-up in same thread) ---")
print("User: How does that compare to its price 6 months ago?\n")
turn_2 = invoke_agent(
    "How does that compare to its price 6 months ago?",
    id_token=id_token,
    thread_id=thread_id,
)

## Cell 10: Langfuse Traces

Langfuse captures full traces of every agent invocation — LLM calls, tool
invocations, and graph transitions. Below we fetch recent traces via the
Langfuse API and display screenshots from `notebooks/images/`.

### Trace Screenshots

The following screenshots were captured from the Langfuse Cloud dashboard
during deployment verification. Place your screenshots in `notebooks/images/`.

![Langfuse Trace Overview](images/langfuse-trace-overview.png)

![Langfuse Trace Detail](images/langfuse-trace-detail.png)

In [ ]:
import os

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST = os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    from langfuse import Langfuse

    lf = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
    )

    traces = lf.fetch_traces(limit=5, order_by="timestamp.desc")
    print(f"Recent Langfuse traces ({len(traces.data)} found):")
    print("-" * 80)
    for t in traces.data:
        print(f"  ID: {t.id}")
        print(f"  Name: {t.name}")
        print(f"  Timestamp: {t.timestamp}")
        print(f"  URL: {LANGFUSE_HOST}/trace/{t.id}")
        print("-" * 80)
else:
    print("Langfuse keys not set — skipping trace retrieval.")
    print("Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to fetch traces.")
    print("\nSee screenshots in notebooks/images/ for trace examples.")